# Notebook 2 — Base Model: Lightweight Temporal–Spatial CNN

**Base model template notebook** for EEG-based neurological disorder detection.

- Focus disorder (example): **Parkinson’s Disease (PD)**
- Input: preprocessed EEG epochs shaped `(N, C, T)`
- Output: class logits for `PD vs Healthy` (binary)

> This notebook is a **code structure template**. Replace the data-loading and preprocessing placeholders with your dataset pipeline.


In [ ]:
# ===== Imports =====
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Optional: EEG preprocessing (use if you load raw EEG here)
# import mne

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:

# ===== Config =====
@dataclass
class Config:
    # Data paths
    data_dir: str = "../data/processed/ds004584"

    # Model parameters (updated after loading data)
    n_channels: int = 63
    n_classes: int = 2
    sampling_rate: int = 250
    epoch_seconds: float = 2.0
    n_samples: int = 500  # 2s × 250Hz, updated after loading

    # Training
    batch_size: int = 32
    lr: float = 5e-4
    weight_decay: float = 5e-4
    epochs: int = 3
    patience: int = 12
    grad_clip_norm: float = 1.0

    # Model
    # IMPROVED: dropout 0.50→0.40 (was over-regularising, killing PD recall)
    dropout: float = 0.40

    # Data augmentation — tuned down slightly to avoid over-augmenting
    use_augmentation: bool = True
    noise_std: float = 0.10
    time_shift_max: int = 15
    scale_range: tuple = (0.90, 1.10)
    time_mask_ratio: float = 0.12
    channel_drop_prob: float = 0.08

    # Loss function
    # IMPROVED: Focal Loss + large PD weight boost.
    # The previous run had Control=1.2, PD=1.0 — PD was weighted *lower*
    # than Control, which directly caused recall=0.60.
    use_focal_loss: bool = True
    focal_gamma: float = 2.0
    pd_weight_boost: float = 2.5

    # Weighted sampler to oversample PD in each batch
    use_weighted_sampler: bool = True
    sampler_pd_boost: float = 2.0

    # Threshold tuning settings
    min_pd_recall: float = 0.75
    min_control_recall: float = 0.65


cfg = Config()
print(f"dropout={cfg.dropout}, focal_gamma={cfg.focal_gamma}, pd_weight_boost={cfg.pd_weight_boost}")
print(f"epochs={cfg.epochs}, patience={cfg.patience}, lr={cfg.lr}, batch={cfg.batch_size}")
print(f"sampler_pd_boost={cfg.sampler_pd_boost}, time_mask={cfg.time_mask_ratio}, ch_drop={cfg.channel_drop_prob}")
print(f"Target: PD Recall >={cfg.min_pd_recall:.0%}, Control Recall >={cfg.min_control_recall:.0%}")


## Data loading

Recommended approach:

1. **Preprocess raw EEG** using MNE (filter, notch, ICA, re-reference).
2. **Epoch/segment** into fixed windows (e.g., 2s).
3. Save as `npz` with:
   - `X`: float32 array `(N, C, T)`
   - `y`: int64 array `(N,)` labels (0=HC, 1=PD)
   - optional: `subject_id`: `(N,)` to do subject-wise splits.


In [ ]:
# ===== Load preprocessed data (train/val/test splits) =====
def load_split(data_dir: str, split: str):
    """Load a data split (train/val/test) from .npz file."""
    path = Path(data_dir) / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected preprocessed data at {path}.\n"
            "Run preprocessing/preprocess_ds004584.ipynb first to generate the data."
        )
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)  # (N, C, T)
    y = data["y"].astype(np.int64)    # (N,)
    subject_id = data["subject_id"] if "subject_id" in data.files else None
    return X, y, subject_id

# Load all splits
print("Loading preprocessed data...")
# Resolve data directory robustly (handles different notebook working directories)
candidate_dirs = [
    Path(cfg.data_dir),
    (Path.cwd() / cfg.data_dir).resolve(),
    (Path.cwd() / ".." / "data" / "processed" / "ds004584").resolve(),
    (Path.cwd() / "data" / "processed" / "ds004584").resolve(),
]

resolved_data_dir = next((p for p in candidate_dirs if (p / "train.npz").exists()), None)
if resolved_data_dir is None:
    # Extra fallback search: walk up parent folders and try <parent>/data/processed/ds004584
    fallback_dirs = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = (base / "data" / "processed" / "ds004584").resolve()
        fallback_dirs.append(p)
        if (p / "train.npz").exists():
            resolved_data_dir = p
            print(f"Found dataset via fallback search: {resolved_data_dir}")
            break

    if resolved_data_dir is None:
        checked = []
        for p in [*candidate_dirs, *fallback_dirs]:
            sp = str(p)
            if sp not in checked:
                checked.append(sp)

        raise FileNotFoundError(
            "Could not find train.npz in any expected location.\n"
            f"Current working directory: {Path.cwd()}\n"
            "Checked:\n- " + "\n- ".join(checked) +
            "\nRun preprocessing/preprocess_ds004584.ipynb first."
        )

cfg.data_dir = str(resolved_data_dir)
print(f"Using data_dir: {cfg.data_dir}")

X_train, y_train, sid_train = load_split(cfg.data_dir, "train")
X_val, y_val, sid_val = load_split(cfg.data_dir, "val")
X_test, y_test, sid_test = load_split(cfg.data_dir, "test")

# Update config with actual dimensions
cfg.n_channels = X_train.shape[1]
n_samples = X_train.shape[2]

print(f"\n{'='*50}")
print("DATASET LOADED")
print(f"{'='*50}")
print(f"Train: {X_train.shape} | PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} | PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} | PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")
print(f"\nChannels: {cfg.n_channels}, Samples per epoch: {n_samples}")

In [ ]:

# ===== Torch Dataset with Augmentation =====
class EEGDataset(Dataset):
    def __init__(self, X, y, augment=False, cfg=None):
        self.X = torch.from_numpy(X)  # (N, C, T)
        self.y = torch.from_numpy(y)  # (N,)
        self.augment = augment
        self.cfg = cfg

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment and self.cfg is not None:
            # Gaussian noise
            x = x + torch.randn_like(x) * self.cfg.noise_std
            # Random amplitude scaling
            scale = self.cfg.scale_range[0] + (self.cfg.scale_range[1] - self.cfg.scale_range[0]) * torch.rand(1).item()
            x = x * scale
            # Random time shift
            if self.cfg.time_shift_max > 0:
                shift = torch.randint(-self.cfg.time_shift_max, self.cfg.time_shift_max + 1, (1,)).item()
                if shift != 0:
                    x = torch.roll(x, shifts=shift, dims=-1)
            # Time masking (NEW)
            if self.cfg.time_mask_ratio > 0:
                T = x.shape[-1]
                mask_len = int(T * self.cfg.time_mask_ratio)
                if mask_len > 0:
                    start = torch.randint(0, T - mask_len, (1,)).item()
                    x = x.clone()
                    x[..., start:start + mask_len] = 0
            # Channel dropout (NEW)
            if self.cfg.channel_drop_prob > 0:
                C = x.shape[0]
                drop_mask = torch.rand(C) < self.cfg.channel_drop_prob
                if drop_mask.any():
                    x = x.clone()
                    x[drop_mask, :] = 0
        return x, self.y[idx]


def make_loaders(X_train, y_train, X_val, y_val, cfg):
    """Create train/val DataLoaders, with optional WeightedRandomSampler for PD oversampling."""
    from torch.utils.data import WeightedRandomSampler

    train_ds = EEGDataset(X_train, y_train, augment=cfg.use_augmentation, cfg=cfg)
    val_ds = EEGDataset(X_val, y_val, augment=False)

    if cfg.use_weighted_sampler:
        class_counts = np.bincount(y_train)
        weights_np = 1.0 / np.maximum(class_counts, 1).astype(float)
        weights_np[1] *= cfg.sampler_pd_boost  # Extra PD emphasis
        sample_weights = weights_np[y_train]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, drop_last=False)
        print(f"WeightedRandomSampler active (PD boost={cfg.sampler_pd_boost}x, counts={class_counts.tolist()})")
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader


## Model: Lightweight Temporal–Spatial CNN

Structure idea:
- **Temporal Conv** (learn rhythms)
- **Spatial Conv** (learn channel interactions)
- Pooling + dropout
- Linear classifier

This is easy to explain and a strong baseline.

In [ ]:

# ===== Lightweight Temporal–Spatial CNN (improved: 4-scale + tuned capacity) =====
class MultiScaleTemporal(nn.Module):
    """
    Multi-scale temporal convolutions to capture EEG frequency bands:
      short  (k=13)  → beta/gamma (~13-100 Hz)
      medium (k=25)  → alpha/beta (~8-30 Hz)
      long   (k=51)  → theta/alpha (~4-13 Hz)
      xlong  (k=101) → delta/theta (~1-8 Hz)   ← NEW: key for PD slow-wave biomarkers
    """
    def __init__(self, out_per_scale=8):
        super().__init__()
        self.conv_short = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 13), padding=(0, 6), bias=False),
            nn.BatchNorm2d(out_per_scale), nn.ELU()
        )
        self.conv_mid = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(out_per_scale), nn.ELU()
        )
        self.conv_long = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 51), padding=(0, 25), bias=False),
            nn.BatchNorm2d(out_per_scale), nn.ELU()
        )
        self.conv_xlong = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 101), padding=(0, 50), bias=False),
            nn.BatchNorm2d(out_per_scale), nn.ELU()
        )

    def forward(self, x):
        # x: (B, 1, C, T)
        t_min = x.shape[-1]
        short = self.conv_short(x)[..., :t_min]
        mid   = self.conv_mid(x)[..., :t_min]
        long  = self.conv_long(x)[..., :t_min]
        xlong = self.conv_xlong(x)[..., :t_min]
        return torch.cat([short, mid, long, xlong], dim=1)  # (B, 4*out_per_scale, C, T)


class LWTemporalSpatialCNN(nn.Module):
    """
    Improved Lightweight Temporal-Spatial CNN.

    Changes from baseline:
    - 4-scale temporal front (added delta/theta band kernel)
    - dropout from config (reduced to 0.40 to recover PD recall)
    - conf_head for optional error-correction stage
    """
    def __init__(self, n_channels, n_classes, T, dropout=0.40):
        super().__init__()
        out_per_scale = 8
        temporal_out = out_per_scale * 4   # 32 from 4 scales
        spatial_out = 64

        self.temporal = MultiScaleTemporal(out_per_scale=out_per_scale)

        self.spatial = nn.Sequential(
            nn.Conv2d(temporal_out, spatial_out, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(spatial_out),
            nn.ELU()
        )
        self.pool = nn.Sequential(
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )
        self.refine = nn.Sequential(
            nn.Conv2d(spatial_out, spatial_out, kernel_size=(1, 9), padding=(0, 4), bias=False),
            nn.BatchNorm2d(spatial_out),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        # Compute feature dim
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, T)
            out = self.refine(self.pool(self.spatial(self.temporal(dummy))))
            self.feat_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(self.feat_dim, n_classes)
        self.conf_head = nn.Linear(self.feat_dim, 1)

    def forward(self, x, return_features=False):
        x = x.unsqueeze(1)       # (B, C, T) → (B, 1, C, T)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.pool(x)
        x = self.refine(x)
        feats = x.flatten(1)
        logits = self.classifier(feats)
        if return_features:
            conf = torch.sigmoid(self.conf_head(feats))
            return logits, conf, feats
        return logits


T = n_samples
model = LWTemporalSpatialCNN(cfg.n_channels, cfg.n_classes, T=T, dropout=cfg.dropout).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Feature dimension: {model.feat_dim}")


In [ ]:

# ===== Training & Evaluation utilities =====
def train_one_epoch(model, loader, optimizer, criterion, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))


def eval_model(model, loader):
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(pred)
            y_probs.append(probs.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_probs = np.concatenate(y_probs)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_true, y_pred, y_probs


## Training with early stopping

Using the existing subject-wise train/val/test splits from preprocessing.
Training with early stopping on validation F1 score.

In [ ]:

# ===== Training: Focal Loss + WeightedRandomSampler + Gradient Clipping =====

# --- Data loaders ---
train_loader, val_loader = make_loaders(X_train, y_train, X_val, y_val, cfg)
test_ds = EEGDataset(X_test, y_test, augment=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

# --- Focal Loss ---
class FocalLoss(nn.Module):
    """Focal Loss: down-weights easy examples, focuses on hard-to-classify ones."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        fl = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            fl = self.alpha[targets] * fl
        return fl.mean()

# --- Class weights with PD boost ---
# FIXED: previous weights were Control=1.2, PD=1.0 (PD *lower* than Control!)
class_counts = np.bincount(y_train)
cw_np = len(y_train) / (len(class_counts) * class_counts.astype(float))
cw_np[1] *= cfg.pd_weight_boost
class_weights_t = torch.tensor(cw_np, dtype=torch.float32).to(DEVICE)
print(f"Class weights → Control={class_weights_t[0]:.3f}, PD={class_weights_t[1]:.3f}")

if cfg.use_focal_loss:
    criterion = FocalLoss(alpha=class_weights_t, gamma=cfg.focal_gamma)
    print(f"Using Focal Loss (gamma={cfg.focal_gamma})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=0.05)
    print("Using CrossEntropyLoss")

# --- Fresh model + optimizer + scheduler ---
model = LWTemporalSpatialCNN(cfg.n_channels, cfg.n_classes, T=n_samples, dropout=cfg.dropout).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

best_f1 = -1
best_state = None
patience_counter = 0
train_losses, val_accs, val_f1s = [], [], []

print(f"\nTraining for up to {cfg.epochs} epochs (patience={cfg.patience})...")
print(f"Augmentation: {cfg.use_augmentation} | Focal Loss: {cfg.use_focal_loss}")
print("=" * 70)

for epoch in range(1, cfg.epochs + 1):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, grad_clip=cfg.grad_clip_norm)
    acc, f1, cm, _, _, _ = eval_model(model, val_loader)

    train_losses.append(loss)
    val_accs.append(acc)
    val_f1s.append(f1)

    scheduler.step()

    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *** best ***"
    else:
        patience_counter += 1
        marker = ""

    if epoch % 3 == 0 or epoch == 1 or marker:
        print(f"Epoch {epoch:03d} | loss={loss:.4f} | val_acc={acc:.4f} | val_f1={f1:.4f}{marker}")

    if patience_counter >= cfg.patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
        break

model.load_state_dict(best_state)
print(f"\nBest validation F1: {best_f1:.4f}")


## Test Set Evaluation & Visualization

In [ ]:

# ===== Final Test Evaluation with Threshold Tuning =====
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

# --- Tune threshold on validation set ---
model.eval()
val_probs_all, val_labels_all = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        val_probs_all.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        val_labels_all.extend(yb.numpy())
val_probs_all = np.array(val_probs_all)
val_labels_all = np.array(val_labels_all)

best_thr = 0.5
best_bal = -1
for thr in np.arange(0.20, 0.70, 0.01):
    preds_v = (val_probs_all >= thr).astype(int)
    pd_mask = val_labels_all == 1
    ctrl_mask = val_labels_all == 0
    pd_rec = np.mean(preds_v[pd_mask] == 1) if pd_mask.any() else 0.0
    ctrl_rec = np.mean(preds_v[ctrl_mask] == 0) if ctrl_mask.any() else 0.0
    bal = 0.5 * (pd_rec + ctrl_rec)
    if pd_rec >= 0.75 and ctrl_rec >= 0.65 and bal > best_bal:
        best_bal = bal
        best_thr = thr

print(f"Optimal threshold (val): {best_thr:.2f}")

# --- Collect test probabilities ---
all_probs, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE))
        all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        all_labels.extend(yb.numpy())
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# --- Report at both thresholds ---
print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
for thr, label in [(0.5, "default (0.50)"), (best_thr, f"tuned  ({best_thr:.2f})")]:
    y_pred = (all_probs >= thr).astype(int)
    acc = accuracy_score(all_labels, y_pred)
    f1 = f1_score(all_labels, y_pred)
    cm = confusion_matrix(all_labels, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\n[Threshold = {label}]")
    print(f"  Accuracy={acc:.4f}  F1={f1:.4f}")
    print(f"  PD Recall(Sensitivity)={tp/(tp+fn):.4f}   Ctrl Recall(Specificity)={tn/(tn+fp):.4f}")
    print(f"  Confusion Matrix:\n{cm}")

optimal_preds = (all_probs >= best_thr).astype(int)
print(f"\n{classification_report(all_labels, optimal_preds, target_names=['Control', 'PD'], digits=4)}")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(train_losses, label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_accs, label='Val Accuracy', color='blue')
axes[1].plot(val_f1s, label='Val F1', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

cm_plot = confusion_matrix(all_labels, optimal_preds)
ConfusionMatrixDisplay(cm_plot, display_labels=['Control', 'PD']).plot(ax=axes[2], cmap='Blues')
axes[2].set_title(f'Test Confusion Matrix (thr={best_thr:.2f})')

plt.tight_layout()
plt.show()

# --- Save model ---
save_dir = Path("../training/saved_models")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "lw_temporal_spatial_cnn_parkinsons.pth"
torch.save(model.state_dict(), save_path)
print(f"\nModel saved to: {save_path}")


## Hybrid Model: Base LW-TemporalSpatial-CNN + Error Correction Network (ECN)

Train an ECN on top of the best base model checkpoint.

- **Base model** predicts `PD vs Control` from EEG features
- **ECN** learns a residual correction over the base model's features, softmax probabilities, and confidence score
- **Final prediction** uses `base_logits + residual_logits`
- Base model weights are **frozen** during ECN training

In [ ]:

# ===== ECN: Definition + Utilities =====
class ErrorCorrectionNetwork(nn.Module):
    """Residual correction head over frozen base-model features and confidence.

    Input:  [base_features (feat_dim) | softmax_probs (n_classes) | confidence (1)]
    Output: residual logits added to base logits → hybrid logits
    """
    def __init__(self, feat_dim, n_classes=2, hidden_dim=256, dropout=0.20):
        super().__init__()
        input_dim = feat_dim + n_classes + 1
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def _forward_hybrid(base_model, ecn_model, xb):
    base_logits, base_conf, base_feats = base_model(xb, return_features=True)
    base_probs = torch.softmax(base_logits, dim=1)
    ecn_input = torch.cat([base_feats, base_probs, base_conf], dim=1)
    residual_logits = ecn_model(ecn_input)
    hybrid_logits = base_logits + residual_logits
    return base_logits, hybrid_logits, residual_logits


def train_hybrid_ecn(base_model, ecn_model, train_loader, val_loader, cw_tensor,
                     epochs=15, lr=3e-4, residual_penalty=5e-4, patience=5):
    """Train ECN while keeping base_model frozen."""
    base_model.eval()
    for p in base_model.parameters():
        p.requires_grad = False

    optimizer = optim.AdamW(ecn_model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=cw_tensor.detach())

    best_state = None
    best_val_f1 = -1.0
    patience_ctr = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    print("=" * 60)
    print("HYBRID TRAINING (ECN over frozen base)")
    print("=" * 60)

    for epoch in range(1, epochs + 1):
        ecn_model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()

            with torch.no_grad():
                base_logits, base_conf, base_feats = base_model(xb, return_features=True)
                base_probs = torch.softmax(base_logits, dim=1)
                ecn_input = torch.cat([base_feats, base_probs, base_conf], dim=1)

            residual_logits = ecn_model(ecn_input)
            hybrid_logits = base_logits + residual_logits

            cls_loss = criterion(hybrid_logits, yb)
            reg_loss = residual_penalty * residual_logits.pow(2).mean()
            loss = cls_loss + reg_loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))

        ecn_model.eval()
        val_loss = 0.0
        val_preds_list, val_labels_list = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                _, hybrid_logits, residual_logits = _forward_hybrid(base_model, ecn_model, xb)
                cls_loss = criterion(hybrid_logits, yb)
                reg_loss = residual_penalty * residual_logits.pow(2).mean()
                val_loss += (cls_loss + reg_loss).item()
                val_preds_list.extend(torch.argmax(hybrid_logits, dim=1).cpu().numpy())
                val_labels_list.extend(yb.cpu().numpy())

        val_loss /= max(1, len(val_loader))
        val_f1 = f1_score(val_labels_list, val_preds_list)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in ecn_model.state_dict().items()}
            patience_ctr = 0
            marker = " *** BEST ***"
        else:
            patience_ctr += 1
            marker = ""

        print(f"ECN Epoch {epoch:03d} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Val F1={val_f1:.4f}{marker}")

        if patience_ctr >= patience:
            print(f"Early stopping ECN at epoch {epoch}")
            break

    if best_state is not None:
        ecn_model.load_state_dict(best_state)
    print(f"Best ECN Val F1: {best_val_f1:.4f}")
    return ecn_model, history


def collect_pd_probs(base_model, loader, ecn_model=None):
    """Collect PD class probabilities from base or hybrid model."""
    base_model.eval()
    if ecn_model is not None:
        ecn_model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            if ecn_model is None:
                logits = base_model(xb)
            else:
                _, logits, _ = _forward_hybrid(base_model, ecn_model, xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(yb.numpy())
    return np.array(all_labels), np.array(all_probs)


def tune_threshold_from_probs(labels, probs, min_pd_recall=0.75, min_control_recall=0.65):
    """Find classification threshold maximising balanced accuracy subject to recall constraints."""
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    results = []
    for thresh in np.arange(0.20, 0.70, 0.01):
        preds = (probs >= thresh).astype(int)
        pd_mask = labels == 1
        ctrl_mask = labels == 0
        pd_recall = np.mean(preds[pd_mask] == 1) if np.any(pd_mask) else 0.0
        ctrl_recall = np.mean(preds[ctrl_mask] == 0) if np.any(ctrl_mask) else 0.0
        f1 = f1_score(labels, preds)
        bal = 0.5 * (pd_recall + ctrl_recall)
        results.append((thresh, f1, pd_recall, ctrl_recall, bal))
    feasible = [r for r in results if r[2] >= min_pd_recall and r[3] >= min_control_recall]
    best = max(feasible if feasible else results, key=lambda r: r[4])
    return best[0], best[1], best[2], best[3]


print("ECN utilities defined: ErrorCorrectionNetwork, train_hybrid_ecn, collect_pd_probs, tune_threshold_from_probs")


In [ ]:

# ===== Train ECN on top of best base checkpoint =====
# Reload the best base model weights
model.load_state_dict(best_state)
model.eval()

ecn_model = ErrorCorrectionNetwork(
    feat_dim=model.feat_dim,
    n_classes=cfg.n_classes,
    hidden_dim=256,
    dropout=0.20,
).to(DEVICE)

ecn_model, ecn_history = train_hybrid_ecn(
    model,
    ecn_model,
    train_loader,
    val_loader,
    cw_tensor=class_weights_t,
    epochs=max(15, cfg.epochs // 2),
    lr=3e-4,
    residual_penalty=5e-4,
    patience=5,
)
print("ECN training complete.")
print(f"ECN parameters: {sum(p.numel() for p in ecn_model.parameters()):,}")


In [ ]:

# ===== Base vs Hybrid Evaluation (ECN) =====
val_labels_base, val_probs_base = collect_pd_probs(model, val_loader, ecn_model=None)
val_labels_hyb, val_probs_hyb = collect_pd_probs(model, val_loader, ecn_model=ecn_model)

base_thr, base_val_f1, base_val_pd_rec, base_val_ctrl_rec = tune_threshold_from_probs(
    val_labels_base, val_probs_base,
    min_pd_recall=cfg.min_pd_recall, min_control_recall=cfg.min_control_recall,
)
hyb_thr, hyb_val_f1, hyb_val_pd_rec, hyb_val_ctrl_rec = tune_threshold_from_probs(
    val_labels_hyb, val_probs_hyb,
    min_pd_recall=cfg.min_pd_recall, min_control_recall=cfg.min_control_recall,
)

print("=" * 60)
print("VALIDATION THRESHOLD TUNING")
print("=" * 60)
print(f"Base   threshold={base_thr:.2f} | F1={base_val_f1:.4f} | PD recall={base_val_pd_rec:.4f} | Ctrl recall={base_val_ctrl_rec:.4f}")
print(f"Hybrid threshold={hyb_thr:.2f} | F1={hyb_val_f1:.4f} | PD recall={hyb_val_pd_rec:.4f} | Ctrl recall={hyb_val_ctrl_rec:.4f}")

# Test set evaluation
test_labels_base, test_probs_base_v = collect_pd_probs(model, test_loader, ecn_model=None)
test_labels_hyb, test_probs_hyb_v = collect_pd_probs(model, test_loader, ecn_model=ecn_model)

test_preds_base_v = (test_probs_base_v >= base_thr).astype(int)
test_preds_hyb_v = (test_probs_hyb_v >= hyb_thr).astype(int)

base_acc_ecn = accuracy_score(test_labels_base, test_preds_base_v)
base_f1_ecn = f1_score(test_labels_base, test_preds_base_v)
hyb_acc_ecn = accuracy_score(test_labels_hyb, test_preds_hyb_v)
hyb_f1_ecn = f1_score(test_labels_hyb, test_preds_hyb_v)

base_wrong = test_preds_base_v != test_labels_base
hyb_wrong = test_preds_hyb_v != test_labels_hyb
rescued = int(np.sum(base_wrong & (~hyb_wrong)))
hurt = int(np.sum((~base_wrong) & hyb_wrong))

test_cm_base_ecn = confusion_matrix(test_labels_base, test_preds_base_v)
test_cm_hyb_ecn = confusion_matrix(test_labels_hyb, test_preds_hyb_v)

print("\n" + "=" * 60)
print("TEST RESULTS: BASE vs HYBRID (ECN)")
print("=" * 60)
print(f"Base   | Acc={base_acc_ecn:.4f} | F1={base_f1_ecn:.4f}")
print(f"Hybrid | Acc={hyb_acc_ecn:.4f} | F1={hyb_f1_ecn:.4f}")
print(f"Delta  | Acc={hyb_acc_ecn - base_acc_ecn:+.4f} | F1={hyb_f1_ecn - base_f1_ecn:+.4f}")
print(f"Rescued errors: {rescued}   |   New errors introduced: {hurt}")

print("\nBase Confusion Matrix:")
tn, fp, fn, tp = test_cm_base_ecn.ravel()
print(f"                  Predicted")
print(f"                  Control   PD")
print(f"  Actual Control   {tn:5d}  {fp:5d}")
print(f"  Actual PD        {fn:5d}  {tp:5d}")

print("\nHybrid Confusion Matrix:")
tn, fp, fn, tp = test_cm_hyb_ecn.ravel()
print(f"                  Predicted")
print(f"                  Control   PD")
print(f"  Actual Control   {tn:5d}  {fp:5d}")
print(f"  Actual PD        {fn:5d}  {tp:5d}")

print(f"\n{'='*60}")
print("CLASSIFICATION REPORT (Hybrid, tuned threshold)")
print(f"{'='*60}")
print(classification_report(test_labels_hyb, test_preds_hyb_v, target_names=["Control (0)", "PD (1)"], digits=4))

# Save hybrid ECN bundle
hybrid_save_path = Path("../training/saved_models") / "lw_temporal_spatial_cnn_hybrid_ecn_parkinsons.pth"
torch.save({
    'base_model_state_dict': best_state,
    'ecn_model_state_dict': ecn_model.state_dict(),
    'config': {'n_channels': cfg.n_channels, 'n_classes': cfg.n_classes,
               'n_samples': n_samples, 'dropout': cfg.dropout},
    'hybrid_results': {
        'base_threshold': float(base_thr), 'hybrid_threshold': float(hyb_thr),
        'base_f1': float(base_f1_ecn), 'hybrid_f1': float(hyb_f1_ecn),
        'rescued_errors': rescued, 'new_errors': hurt,
    },
}, hybrid_save_path)
print(f"\nHybrid ECN bundle saved to: {hybrid_save_path}")
